# PDF Processing Pipeline: Complete Workflow

This notebook processes scientific PDFs through table reconstruction, masking, text extraction, and database ingestion.

## Pipeline Overview:
1. **Setup** - Imports and configuration
2. **Original PDF Processing** - Layout extraction and table reconstruction
3. **PDF Masking** - Remove tables/figures and re-extract layout
4. **Text Processing** - Group by hierarchical path and stitch paragraphs
5. **Media Extraction** - Crop and save tables/figures
6. **Database Ingestion** - Save to PostgreSQL with hierarchical structure

**Date**: 2025-12-31

## 1. Setup

### 1.1 Imports and Module Loading

In [39]:
import sys
import json
from pathlib import Path
import fitz
from IPython.display import Image, display
import pandas as pd
import importlib.util
from collections import Counter
from datetime import datetime

project_root = Path.cwd()
sys.path.insert(0, str(project_root))

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

mask_tables = load_module('mask_tables', project_root / 'scripts/docling/mask_tables.py')
visualize = load_module('visualize', project_root / 'scripts/visualize_docling_full.py')
text_proc = load_module('text_proc', project_root / 'parsers/text_processing.py')

process_pdf_with_masking = mask_tables.process_pdf_with_masking
reconstruct_tables_from_lists = visualize.reconstruct_tables_from_lists
ContextAwareStitcher = text_proc.ContextAwareStitcher
remove_citations = text_proc.remove_citations

print("✅ Imports successful!")

✅ Imports successful!


### 1.2 Configure Paths and Directories

In [40]:
PDF_PATH = Path('files/organized_pdfs/PMC1448691_his_2369.pdf')
PMCID = 'PMC1448691'

DOCLING_OUTPUT_DIR = Path('out/docling_full')
MASKED_PDF_DIR = Path('out/masked_pdfs')
TEXT_OUTPUT_DIR = Path('out/text')
TABLES_DIR = Path('files/tables')
FIGURES_DIR = Path('files/figures')

for d in [DOCLING_OUTPUT_DIR, MASKED_PDF_DIR, TEXT_OUTPUT_DIR, TABLES_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"📄 PDF: {PDF_PATH.name}")
print(f"📁 PMCID: {PMCID}")

📄 PDF: PMC1448691_his_2369.pdf
📁 PMCID: PMC1448691


## 2. Original PDF Processing

### 2.1 Extract Layout with Docling

Extract document structure from the original PDF including text, tables, figures, and bounding boxes.

In [41]:
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat

pipeline_options = PdfPipelineOptions()
pipeline_options.do_table_structure = False
pipeline_options.do_ocr = True
pipeline_options.images_scale = 2.0

converter = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)}
)

print(f"🔄 Extracting layout from ORIGINAL PDF...")
result = converter.convert(str(PDF_PATH))
doc = result.document

all_elements = []
for element, level in doc.iterate_items():
    label = str(getattr(element, "label", "UNKNOWN")).split('.')[-1].upper()
    if not (hasattr(element, 'prov') and element.prov):
        continue
    prov = element.prov[0]
    bbox = prov.bbox
    text = ""
    if hasattr(element, 'text'):
        text = element.text
    elif hasattr(element, 'caption') and element.caption:
        text = element.caption.text
    all_elements.append({
        "type": label,
        "page": prov.page_no,
        "level": level,
        "bbox": {"x1": bbox.l, "y1": bbox.t, "x2": bbox.r, "y2": bbox.b},
        "text": text.strip() if text else None
    })

docling_json_path = DOCLING_OUTPUT_DIR / f"{PDF_PATH.stem}_full_layout.json"
with open(docling_json_path, 'w') as f:
    json.dump({
        "metadata": {"pdf_path": str(PDF_PATH), "tool": "Docling", "extraction_date": datetime.now().isoformat()},
        "page_dimensions": {no: {"width": p.size.width, "height": p.size.height} for no, p in doc.pages.items()},
        "elements": all_elements
    }, f, indent=2)

types = Counter([el['type'] for el in all_elements])
print(f"\n✅ Original PDF: {len(all_elements)} elements")
print(f"   Tables: {types.get('TABLE', 0)}")
print(f"   Figures: {types.get('PICTURE', 0)}")
print(f"   Captions: {types.get('CAPTION', 0)}")
print(f"💾 {docling_json_path}")

INFO: detected formats: [<InputFormat.PDF: 'pdf'>]
INFO: Going to convert document batch...
INFO: Initializing pipeline for StandardPdfPipeline with options hash 06b2cfbc41861ca6858eec36b7d53267
INFO: Auto OCR model selected ocrmac.
INFO: Accelerator device: 'mps'


🔄 Extracting layout from ORIGINAL PDF...


INFO: Processing document PMC1448691_his_2369.pdf
INFO: Finished converting document PMC1448691_his_2369.pdf in 51.01 sec.



✅ Original PDF: 422 elements
   Tables: 2
   Figures: 9
   Captions: 14
💾 out/docling_full/PMC1448691_his_2369_full_layout.json


### 2.2 Reconstruct Tables from Captions

Group table captions with their content rows to create unified table elements.

In [42]:
print("🔄 Reconstructing tables...")
reconstructed_elements = reconstruct_tables_from_lists(str(docling_json_path))
reconstructed_tables = [el for el in reconstructed_elements if el.get('type') == 'RECONSTRUCTED_TABLE']
print(f"✅ Created {len(reconstructed_tables)} reconstructed tables")

🔄 Reconstructing tables...
✅ Created 4 reconstructed tables


## 3. PDF Masking

### 3.1 Create Masked PDF

Replace tables and figures with white rectangles to create a text-only PDF.

In [43]:
print("🔄 Masking tables and figures...")
masked_pdf_path, _, masked_elements = process_pdf_with_masking(
    pdf_path=PDF_PATH,
    json_path=docling_json_path,
    output_dir=MASKED_PDF_DIR
)
print(f"\n✅ Masked PDF: {masked_pdf_path}")
print(f"🚫 Masked: {len(masked_elements)} elements (tables/figures/captions)")

INFO: Processing: PMC1448691_his_2369.pdf
INFO:   Reconstructing tables from elements...
INFO:   Found 29 maskable elements (including 4 reconstructed tables)
INFO:     ✓ Masked element CAPTION with text = Table 1. B-cell cutaneous lymphoma. Learning from the Workshop on page 2
INFO:     ✓ Masked element RECONSTRUCTED_TABLE with text = NO_TEXT on page 2
INFO:     ✓ Masked element PICTURE with text = None on page 3
INFO:     ✓ Masked element CAPTION with text = Figure 1. a-f, Primary cutaneous follicle centre lymphoma. a, Nodular pattern. b, Centroblastic predominance. c, CD20. d, CD10. e, Bcl-2. f, Ki67. g-j, Primary cutaneous diffuse large B-cell lymphoma, leg type. g,h, morphology, H&E. i, CD20. j, Ki67. Cases contributed by C. Girardet (A6) and R. S. Robertorye (A8). on page 3


🔄 Masking tables and figures...


INFO:     ✓ Masked element TABLE with text = None on page 5
INFO:     ✓ Masked element CAPTION with text = Table 2. Comparison of the immunophenotype of normal and tumoral plasmacytoid dendritic cells (PDC) on page 5
INFO:     ✓ Masked element CAPTION with text = Table 3. Thyroid lymphoma, pointers from the Workshop on page 6
INFO:     ✓ Masked element RECONSTRUCTED_TABLE with text = NO_TEXT on page 6
INFO:     ✓ Masked element PICTURE with text = None on page 7
INFO:     ✓ Masked element CAPTION with text = Figure 2. Follicular lymphoma of the thyroid. a, The thyroid in this 70-year-old male is extensively infiltrated by numerous lymphoid follicles with relatively homogeneous-appearing follicular / germinal centres. b, The interfollicular regions (left) adjacent to the neoplastic-appearing follicle demonstrate many small lymphocytes and prominent lymphoepithelial lesions. c, There are numerous small angulated lymphocytes with pale cytoplasm but only infrequent centroblasts in this fol


✅ Masked PDF: out/masked_pdfs/PMC1448691_his_2369_masked.pdf
🚫 Masked: 29 elements (tables/figures/captions)


### 3.2 Extract Layout from Masked PDF

Run Docling on the masked PDF to extract clean text elements without tables/figures.

In [44]:
print(f"🔄 Extracting layout from MASKED PDF...")
result_masked = converter.convert(str(masked_pdf_path))
doc_masked = result_masked.document

masked_pdf_elements = []
for element, level in doc_masked.iterate_items():
    label = str(getattr(element, "label", "UNKNOWN")).split('.')[-1].upper()
    if not (hasattr(element, 'prov') and element.prov):
        continue
    prov = element.prov[0]
    bbox = prov.bbox
    text = ""
    if hasattr(element, 'text'):
        text = element.text
    elif hasattr(element, 'caption') and element.caption:
        text = element.caption.text
    masked_pdf_elements.append({
        "type": label,
        "page": prov.page_no,
        "level": level,
        "bbox": {"x1": bbox.l, "y1": bbox.t, "x2": bbox.r, "y2": bbox.b},
        "text": text.strip() if text else None
    })

masked_json_path = DOCLING_OUTPUT_DIR / f"{masked_pdf_path.stem}_full_layout.json"
with open(masked_json_path, 'w') as f:
    json.dump({
        "metadata": {"pdf_path": str(masked_pdf_path), "tool": "Docling", "extraction_date": datetime.now().isoformat()},
        "page_dimensions": {no: {"width": p.size.width, "height": p.size.height} for no, p in doc_masked.pages.items()},
        "elements": masked_pdf_elements
    }, f, indent=2)

masked_types = Counter([el['type'] for el in masked_pdf_elements])
print(f"\n✅ Masked PDF: {len(masked_pdf_elements)} elements")
print(f"\n📊 COMPARISON:")
print(f"   Original PDF: {len(all_elements)} elements")
print(f"   Masked PDF:   {len(masked_pdf_elements)} elements")
print(f"   Removed:      {len(all_elements) - len(masked_pdf_elements)} elements")
print(f"\n🔍 Masked PDF element types:")
for t, c in masked_types.most_common():
    print(f"   {t}: {c}")
print(f"\n💾 {masked_json_path}")

# Extract text elements from masked PDF for stitching
text_element_types = {'TEXT', 'PARAGRAPH', 'SECTION_HEADER', 'TITLE', 'LIST', 'LIST_ITEM'}
text_elements = [el for el in masked_pdf_elements if el.get('type') in text_element_types]
print(f"\n📝 Text elements for processing: {len(text_elements)}")

INFO: detected formats: [<InputFormat.PDF: 'pdf'>]
INFO: Going to convert document batch...
INFO: Processing document PMC1448691_his_2369_masked.pdf


🔄 Extracting layout from MASKED PDF...


INFO: Finished converting document PMC1448691_his_2369_masked.pdf in 31.44 sec.



✅ Masked PDF: 284 elements

📊 COMPARISON:
   Original PDF: 422 elements
   Masked PDF:   284 elements
   Removed:      138 elements

🔍 Masked PDF element types:
   LIST_ITEM: 153
   TEXT: 111
   SECTION_HEADER: 19
   FOOTNOTE: 1

💾 out/docling_full/PMC1448691_his_2369_masked_full_layout.json

📝 Text elements for processing: 283


## 4. Text Processing

### 4.1 Group Text by Hierarchical Path

Combine text elements that belong to the same section/subsection.

In [45]:
from collections import defaultdict

# Build hierarchical paths by tracking section headers
hierarchy_tracker = {}  # level -> section name
text_by_path = defaultdict(list)

for el in text_elements:
    level = el.get('level', 0)
    text = el.get('text', '').strip()
    if not text:
        continue
    
    # Update hierarchy when we see section headers
    if el.get('type') == 'SECTION_HEADER':
        hierarchy_tracker[level] = text
        # Clear deeper levels when we encounter a new section at this level
        hierarchy_tracker = {k: v for k, v in hierarchy_tracker.items() if k <= level}
    else:
        # Build path from current hierarchy
        path_parts = [hierarchy_tracker.get(l, '') for l in sorted(hierarchy_tracker.keys()) if hierarchy_tracker.get(l)]
        path_string = ' > '.join(path_parts) if path_parts else 'Root'
        
        # Add text to this path
        text_by_path[path_string].append(text)

print(f"📂 Found {len(text_by_path)} unique hierarchical paths")
print(f"\n🔍 Sample paths:")
for i, (path, texts) in enumerate(list(text_by_path.items())[:5]):
    print(f"   {i+1}. {path}: {len(texts)} elements")

# Group and stitch text for each path
stitcher = ContextAwareStitcher()
stitched_by_path = {}

for path, texts in text_by_path.items():
    # Remove citations from each text element
    clean_texts = [remove_citations(t) for t in texts]
    # Stitch paragraphs within this section
    stitched = stitcher.reconstruct_paragraphs(clean_texts)
    stitched_by_path[path] = stitched

total_original = sum(len(v) for v in text_by_path.values())
total_stitched = sum(len(v) for v in stitched_by_path.values())
print(f"\n✂️ Stitching within paths: {total_original} → {total_stitched} paragraphs")
print(f"   Merged: {total_original - total_stitched} split paragraphs")

📂 Found 17 unique hierarchical paths

🔍 Sample paths:
   1. Update on extranodal lymphomas. Conclusions of the Workshop held by the EAHP and the SH in Thessaloniki, Greece: 9 elements
   2. Introduction: 3 elements
   3. Highlights of cutaneous lymphomas: 12 elements
   4. Plasmacytoid dendritic cell tumours: 13 elements
   5. Thyroid lymphoma: 9 elements

✂️ Stitching within paths: 264 → 229 paragraphs
   Merged: 35 split paragraphs


### 4.2 Save Clean Text

Export stitched paragraphs to text file for analysis or database ingestion.

In [46]:
text_path = TEXT_OUTPUT_DIR / f"{PMCID}_stitched.txt"
with open(text_path, 'w') as f:
    f.write(f"Document: {PMCID}\n{'='*80}\n\n")
    
    # Write text organized by hierarchical path
    for path, paragraphs in sorted(stitched_by_path.items()):
        f.write(f"[{path}]\n")
        f.write(f"{'-'*80}\n")
        for p in paragraphs:
            if p.strip():
                f.write(f"{p}\n\n")
        f.write("\n")

print(f"💾 {text_path} ({text_path.stat().st_size / 1024:.1f} KB)")
print(f"📁 Organized by {len(stitched_by_path)} hierarchical paths")

💾 out/text/PMC1448691_stitched.txt (90.0 KB)
📁 Organized by 17 hierarchical paths


## 5. Media Extraction

### 5.1 Crop Table and Figure Images

Extract table and figure regions from the original PDF as high-resolution images.

In [47]:
table_data, figure_data = [], []
tc, fc = 1, 1

for el in masked_elements:
    t = el.get('type')
    if t in ['TABLE', 'RECONSTRUCTED_TABLE']:
        table_data.append({'table_id': str(tc), 'caption': el.get('text', f'Table {tc}'),
                          'page': el.get('page'), 'bbox': el.get('bbox'), 'type': t.lower()})
        tc += 1
    elif t in ['FIGURE', 'PICTURE']:
        figure_data.append({'figure_id': str(fc), 'caption': el.get('text', f'Figure {fc}'),
                           'page': el.get('page'), 'bbox': el.get('bbox'), 'type': t.lower()})
        fc += 1

doc = fitz.open(str(PDF_PATH))
for t in table_data:
    if t['page'] and t['bbox']:
        p = doc[t['page'] - 1]
        h = p.rect.height
        b = t['bbox']
        r = fitz.Rect(b['x1'], h - max(b['y1'], b['y2']), b['x2'], h - min(b['y1'], b['y2']))
        pix = p.get_pixmap(clip=r, matrix=fitz.Matrix(2, 2))
        path = TABLES_DIR / f"{PMCID}_table_{t['table_id']}.png"
        pix.save(str(path))
        t['image_path'] = str(path)

for f in figure_data:
    if f['page'] and f['bbox']:
        p = doc[f['page'] - 1]
        h = p.rect.height
        b = f['bbox']
        r = fitz.Rect(b['x1'], h - max(b['y1'], b['y2']), b['x2'], h - min(b['y1'], b['y2']))
        pix = p.get_pixmap(clip=r, matrix=fitz.Matrix(2, 2))
        path = FIGURES_DIR / f"{PMCID}_figure_{f['figure_id']}.png"
        pix.save(str(path))
        f['image_path'] = str(path)

doc.close()
print(f"🖼️ Cropped {len(table_data)} tables, {len(figure_data)} figures")

🖼️ Cropped 6 tables, 9 figures


### 5.2 Save Metadata

Export table and figure metadata as JSON files.

In [48]:
if table_data:
    with open(TABLES_DIR / f"{PMCID}_tables.json", 'w') as f:
        json.dump(table_data, f, indent=2)
if figure_data:
    with open(FIGURES_DIR / f"{PMCID}_figures.json", 'w') as f:
        json.dump(figure_data, f, indent=2)
print("💾 Metadata saved")

💾 Metadata saved


## 7. Database Ingestion

### 7.1 Prepare Data for Database

Convert stitched text by path into database-compatible hierarchical structure.

In [ ]:
# Import database modules
from database import get_db_connection, Document, TextElement, Figure, Table
from database.models import TextElementFigureReference, TextElementTableReference

# Prepare hierarchical elements for database
# Need to convert stitched_by_path back into individual text elements with hierarchical info
db_text_elements = []

for path_string, stitched_paras in stitched_by_path.items():
    # Build path_list from path_string
    if path_string == 'Root':
        path_list = []
        depth = 0
    else:
        path_list = [part.strip() for part in path_string.split(' > ')]
        depth = len(path_list)
    
    # Each stitched paragraph becomes one text element
    for para in stitched_paras:
        if para.strip():
            # Note: We don't have page info or references at this stage
            # In production, you'd track these during the initial grouping
            db_text_elements.append({
                'path_list': path_list,
                'path_string': path_string,
                'depth': depth,
                'text': para,
                'references': {}  # Would be populated if we tracked them
            })

print(f"📦 Prepared {len(db_text_elements)} text elements for database")
print(f"   Organized into {len(stitched_by_path)} hierarchical paths")

# Preview first few elements
print(f"\n🔍 Sample elements:")
for i, elem in enumerate(db_text_elements[:3]):
    print(f"   {i+1}. Path: {elem['path_string']}")
    print(f"      Text: {elem['text'][:80]}...")
    print()

### 7.2 Ingest to PostgreSQL Database

Save document with hierarchical text elements, figures, and tables to database.

In [ ]:
# Database ingestion (similar to comprehensive_ingest.py)
db = get_db_connection()

try:
    with db.session_scope() as session:
        # Check if document already exists
        existing = session.query(Document).filter_by(pmcid=PMCID).first()
        
        if existing:
            print(f"⚠️  Document {PMCID} already exists in database")
            print(f"   Use force=True to re-ingest or delete manually")
            force_reingest = False  # Set to True to delete and re-create
            
            if force_reingest:
                print(f"🗑️  Deleting existing document...")
                session.delete(existing)
                session.flush()
            else:
                print(f"   Skipping ingestion...")
                raise Exception("Document exists - set force_reingest=True to overwrite")
        
        # Create document record
        doc = Document(
            pmcid=PMCID,
            filename=PDF_PATH.name,
            file_path=str(PDF_PATH.absolute()),
            title=f"Document {PMCID}",  # Could extract from PDF if available
            journal=None,
            publication_year=None,
            text_source='pdf'
        )
        session.add(doc)
        session.flush()
        print(f"✅ Created document: {PMCID}")
        
        # Add text elements with position tracking
        path_counts = defaultdict(int)
        for elem in db_text_elements:
            path_string = elem['path_string']
            position = path_counts[path_string]
            path_counts[path_string] += 1
            
            # Create unique path: {pmcid}/{path_string}/{position}
            unique_path = f"{PMCID}/{path_string}/{position}" if path_string else f"{PMCID}/(Root)/{position}"
            
            text_elem = TextElement(
                unique_path=unique_path,
                document_id=doc.id,
                path_list=elem['path_list'],
                path_string=path_string,
                depth=elem['depth'],
                text_content=elem['text'],
                position_in_section=position,
                references=elem.get('references', {})
            )
            session.add(text_elem)
        
        session.flush()
        print(f"✅ Added {len(db_text_elements)} text elements")
        
        # Add figures
        for fig in figure_data:
            image_filename = None
            image_path = fig.get('image_path')
            if image_path:
                image_filename = Path(image_path).name
            
            figure = Figure(
                document_id=doc.id,
                figure_id=fig['figure_id'],
                figure_label=f"Figure {fig['figure_id']}",
                figure_number=fig['figure_id'],
                caption_text=fig.get('caption'),
                image_filename=image_filename,
                image_path=image_path
            )
            session.add(figure)
        
        session.flush()
        print(f"✅ Added {len(figure_data)} figures")
        
        # Add tables
        for tbl in table_data:
            image_filename = None
            image_path = tbl.get('image_path')
            if image_path:
                image_filename = Path(image_path).name
            
            table = Table(
                document_id=doc.id,
                table_id=tbl['table_id'],
                table_label=f"Table {tbl['table_id']}",
                table_number=tbl['table_id'],
                caption_text=tbl.get('caption'),
                image_filename=image_filename,
                image_path=image_path
            )
            session.add(table)
        
        session.flush()
        print(f"✅ Added {len(table_data)} tables")
        
        # Note: Figure/table references would be created here if we tracked them
        # See comprehensive_ingest.py lines 926-968 for reference creation logic
        
        print(f"\n🎉 Successfully ingested {PMCID} to database!")
        print(f"   Document ID: {doc.id}")
        
except Exception as e:
    print(f"❌ Error: {e}")
    import traceback
    traceback.print_exc()

## 8. Summary

### 8.1 Pipeline Statistics and Output Files

In [ ]:
print("="*80)
print("📊 PIPELINE COMPLETE")
print("="*80)
print(f"\n📄 Input: {PDF_PATH.name}")
print(f"   PMCID: {PMCID}")
print(f"\n📊 Processing Statistics:")
print(f"   Original PDF elements:  {len(all_elements)}")
print(f"   Masked PDF elements:    {len(masked_pdf_elements)}")
print(f"   Elements removed:       {len(all_elements) - len(masked_pdf_elements)}")
print(f"   Text elements extracted: {len(text_elements)}")
print(f"   Hierarchical paths:     {len(stitched_by_path)}")
print(f"   Stitched paragraphs:    {total_stitched}")
print(f"   DB text elements:       {len(db_text_elements)}")
print(f"\n📝 Output Files:")
print(f"   Original JSON:     {docling_json_path.name}")
print(f"   Masked PDF:        {masked_pdf_path.name}")
print(f"   Masked PDF JSON:   {masked_json_path.name}")
print(f"   Text file:         {text_path.name} ({text_path.stat().st_size / 1024:.1f} KB)")
print(f"   Tables metadata:   {TABLES_DIR / f'{PMCID}_tables.json'}")
print(f"   Figures metadata:  {FIGURES_DIR / f'{PMCID}_figures.json'}")
print(f"\n🖼️  Media Extraction:")
print(f"   Tables:    {len(table_data)} images")
print(f"   Figures:   {len(figure_data)} images")
print(f"\n💾 Database Ingestion:")
print(f"   Documents:      1")
print(f"   Text elements: {len(db_text_elements)}")
print(f"   Figures:       {len(figure_data)}")
print(f"   Tables:        {len(table_data)}")
print("\n✅ Done!")
print("\n💡 Key Features:")
print("   ✓ Hierarchical text organization by section paths")
print("   ✓ Paragraph stitching to join split text")
print("   ✓ Citation removal from text content")
print("   ✓ Table reconstruction from captions")
print("   ✓ Clean text extraction via PDF masking")
print("   ✓ PostgreSQL database storage with full relationships")